In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

/Users/amnazafar/opt/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
class CheXpertDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

        self.label_columns = [
            'Atelectasis','Cardiomegaly','Consolidation','Edema',
            'Enlarged Cardiomediastinum','Fracture','Lung Lesion',
            'Lung Opacity','No Finding','Pleural Effusion',
            'Pleural Other','Pneumonia','Pneumothorax','Support Devices'
        ]

        self.df[self.label_columns] = self.df[self.label_columns].fillna(0)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        path = row["Path"]

        # hvis CSV peker til train → bruk valid
        if "train" in path:
            path = path.replace("train", "valid")

        img_path = os.path.join(
            self.img_dir,
            path.replace("CheXpert-v1.0/", "CheXpert-v1.0 batch 1 (validate & csv)/")
        )

        img = Image.open(img_path).convert("RGB")

        labels = row[self.label_columns].values.astype(np.float32)
        labels = torch.tensor(labels)

        if self.transform:
            img = self.transform(img)

        return img, labels

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [4]:
csv_path = "data/chexpert/CheXpert-v1.0 batch 1 (validate & csv)/valid.csv"
img_dir = "data/chexpert"

df = pd.read_csv(csv_path)

dataset = CheXpertDataset(df, img_dir, transform)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [5]:
model = models.densenet121(pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, 14)

/Users/amnazafar/opt/anaconda3/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/amnazafar/opt/anaconda3/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [7]:
epochs = 5  

model.train()

for epoch in range(epochs):
    total_loss = 0

    for images, labels in dataloader:
        labels = labels.float()

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 10.0713
Epoch 2, Loss: 6.6899
Epoch 3, Loss: 4.9323
Epoch 4, Loss: 3.7399
Epoch 5, Loss: 3.0520


In [8]:
torch.save(model.state_dict(), "densenet_chexpert.pth")
print("Model saved ✅")

Model saved ✅
